# EfficientNet-B0 weighted CrossEntropy — 5-fold cross-validation

This notebook runs a 5-fold cross-validation on the full labeled dataset using EfficientNet-B0 with ImageNet pretrained weights and weighted CrossEntropy loss.

The goal is to check whether the good result obtained by the previus experiments with a single train/validation split is stable across different validation partitions, or whether it may depend on a favorable split.

Weighted CrossEntropy is used to reduce the dominance of frequent classes during optimization. The class weights are recomputed inside each fold using only the training subset of that fold.

It calculates the validation balanced accuracy for every fold, then the best model is selected as the model trained on the fold with val bal acc closest to the mean val bal acc of the 5 folds.

## 1. Colab setup, imports and paths

The dataset is stored as a ZIP file in Google Drive. We extract it locally because reading thousands of images directly from Drive is much slower during training.

**Important:** before running the notebook, adapt the paths to your own Google Drive folder.


In [ ]:
from pathlib import Path
import random
import time
import json
import gc
import zipfile

import numpy as np
import pandas as pd

from PIL import Image, ImageOps

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm

from google.colab import drive


drive.mount("/content/drive")

# Google Drive paths.
DRIVE_ROOT = Path("/content/drive/MyDrive/Machine_learning")
DATASET_ZIP_PATH = DRIVE_ROOT / "waste_type_identification.zip"
PROJECT_ROOT = DRIVE_ROOT / "waste_project"

# Local Colab extraction paths.
LOCAL_ROOT = Path("/content")
DATASET_ROOT = LOCAL_ROOT / "waste_type_identification"

# Output folders.
LOG_DIR = PROJECT_ROOT / "logs" / "efficientnet_b0_weighted_ce_5fold"
FIGURE_DIR = PROJECT_ROOT / "figures" / "efficientnet_b0_weighted_ce_5fold"

for directory in [LOG_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Final outputs.
EXPERIMENT_ID = "efficientnet_b0_weighted_ce_5fold_colab"
SUMMARY_PATH = LOG_DIR / f"{EXPERIMENT_ID}_summary.json"
CONFUSION_MATRIX_NORM_PATH = LOG_DIR / f"{EXPERIMENT_ID}_selected_val_confusion_matrix_normalized.csv"
CONFUSION_MATRIX_FIG_PATH = FIGURE_DIR / f"{EXPERIMENT_ID}_selected_val_confusion_matrix_normalized.png"

# Extract the dataset only if it is not already available in the local runtime.
if not DATASET_ROOT.exists():
    if not DATASET_ZIP_PATH.exists():
        raise FileNotFoundError(f"Dataset ZIP not found: {DATASET_ZIP_PATH}")

    with zipfile.ZipFile(DATASET_ZIP_PATH, "r") as zip_file:
        zip_file.extractall(LOCAL_ROOT)

if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Extracted dataset folder not found: {DATASET_ROOT}")

print(f"Dataset root: {DATASET_ROOT}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Summary path: {SUMMARY_PATH}")
print(f"Normalized confusion matrix path: {CONFUSION_MATRIX_NORM_PATH}")


## 2. Experiment constants and reproducibility

This cell defines the project paths, output files, labels, seed and device.
The experiment uses its own identifier so that baseline EfficientNet outputs are not overwritten.


In [ ]:
# Fixed seed for reproducibility.
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Deterministic settings reduce run-to-run variation.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# Cross-validation setup.
K_FOLDS = 5

# Fixed label mapping.
LABEL_TO_CLASS = {
    0: "Battery",
    1: "Clothing",
    2: "Glass",
    3: "Metal",
    4: "Organic",
    5: "Papery",
    6: "Plastic",
    7: "Undifferentiated",
}

# Folder aliases used to infer labels from the dataset structure.
CLASS_ALIASES = {
    0: ["battery", "batteries"],
    1: ["clothing", "clothes", "cloth", "shoe", "shoes"],
    2: ["glass", "brown", "green", "transparent"],
    3: ["metal", "metals"],
    4: ["organic", "organics"],
    5: ["papery", "paper", "cardboard", "cardboards"],
    6: ["plastic", "plastics"],
    7: ["undifferentiated", "other", "others"],
}

ALIAS_TO_LABEL = {
    alias: label
    for label, aliases in CLASS_ALIASES.items()
    for alias in aliases
}

NUM_CLASSES = len(LABEL_TO_CLASS)

# Device configuration.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print(f"Experiment: {EXPERIMENT_ID}")
print(f"Device: {DEVICE}")
print(f"AMP enabled: {USE_AMP}")


## 3. Dataset indexing

We use the full labeled dataset for stratified 5-fold cross-validation. We use a function to infer each label from the folder names.


In [ ]:
def normalize_path_part(text: str) -> str:
    """Normalize a folder name before matching it with class aliases."""
    return text.strip().lower().replace("_", " ").replace("-", " ")


def infer_label_from_relative_path(relative_path: str):
    """Infer the project label by checking the parent folders of an image."""
    parts = Path(relative_path).parts[:-1]

    # Search from the closest folder to the root folder.
    for part in reversed(parts):
        normalized_part = normalize_path_part(part)
        if normalized_part in ALIAS_TO_LABEL:
            return ALIAS_TO_LABEL[normalized_part]

    return None


def build_full_dataset_dataframe(dataset_root: Path) -> pd.DataFrame:
    """Create a dataframe with all labeled images under dataset_root."""
    image_paths = sorted(
        path for path in dataset_root.rglob("*")
        if path.is_file() and path.suffix.lower() == ".jpg"
    )

    rows = []
    unresolved_paths = []

    for image_path in image_paths:
        relative_path = image_path.relative_to(dataset_root).as_posix()
        label = infer_label_from_relative_path(relative_path)

        if label is None:
            unresolved_paths.append(relative_path)
            continue

        rows.append(
            {
                "relative_path": relative_path,
                "label": int(label),
                "class_name": LABEL_TO_CLASS[int(label)],
            }
        )

    if not rows:
        raise ValueError("No labeled images were found. Check DATASET_ROOT.")

    if unresolved_paths:
        examples = "\n".join(unresolved_paths[:20])
        raise ValueError(
            "Some images could not be assigned to a class. "
            f"First unresolved paths:\n{examples}"
        )

    dataframe = pd.DataFrame(rows)
    dataframe = dataframe.drop_duplicates(subset=["relative_path"]).reset_index(drop=True)
    return dataframe


def class_distribution(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Return the number of samples for each class."""
    counts = dataframe["label"].value_counts().sort_index()

    return pd.DataFrame(
        {
            "label": list(range(NUM_CLASSES)),
            "class_name": [LABEL_TO_CLASS[i] for i in range(NUM_CLASSES)],
            "count": [int(counts.get(i, 0)) for i in range(NUM_CLASSES)],
        }
    )


dataset_df = build_full_dataset_dataframe(DATASET_ROOT)
dataset_distribution_df = class_distribution(dataset_df)
min_class_count = int(dataset_distribution_df["count"].min())

print(f"Images used for 5-fold cross-validation: {len(dataset_df):,}")
display(dataset_distribution_df)

# Stratified K-fold needs at least K samples in every class.
if min_class_count < K_FOLDS:
    raise ValueError(
        f"Stratified {K_FOLDS}-fold cross-validation requires at least "
        f"{K_FOLDS} samples per class. The smallest class has {min_class_count} samples."
    )


## 4. Preprocessing

We use the deterministic EfficientNet-B0 preprocessing


In [ ]:
INPUT_SIZE = 224

# EfficientNet-B0 pretrained weights and IMAGENET normalization.
weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
IMAGENET_MEAN = weights.transforms().mean
IMAGENET_STD = weights.transforms().std

# Deterministic preprocessing for all folds.
base_transform = transforms.Compose(
    [
        transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

train_transform = base_transform
val_transform = base_transform

print(f"Input size: {INPUT_SIZE}x{INPUT_SIZE}")
print(f"ImageNet mean: {IMAGENET_MEAN}")
print(f"ImageNet std:  {IMAGENET_STD}")
print("Augmentation: disabled")


## 5. Dataset and DataLoader

Images are loaded from the local Colab folder. The training loader is shuffled inside each fold, while validation is deterministic.


In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 1


class WasteImageDataset(Dataset):
    """Lazy image dataset for waste classification."""

    def __init__(self, dataframe: pd.DataFrame, dataset_root: Path, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.dataset_root = dataset_root
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index: int):
        row = self.dataframe.iloc[index]
        image_path = self.dataset_root / row["relative_path"]

        # Load one image from disk and convert it to RGB.
        with Image.open(image_path) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        label = int(row["label"])
        return image, label


def make_loader(dataframe: pd.DataFrame, transform, shuffle: bool, seed: int = SEED) -> DataLoader:
    """Create a DataLoader for one fold split."""
    dataset = WasteImageDataset(dataframe, DATASET_ROOT, transform=transform)

    generator = None
    if shuffle:
        generator = torch.Generator()
        generator.manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        generator=generator,
    )


print(f"Batch size: {BATCH_SIZE}")
print(f"Number of workers: {NUM_WORKERS}")


## 6. Model, class weights and training settings

EfficientNet-B0 is fully fine-tuned in every fold. Class weights are recomputed from the training subset of each fold, so validation samples do not influence the loss.


In [ ]:
def build_efficientnet_b0_model(num_classes: int) -> nn.Module:
    """Build EfficientNet-B0 with an 8-class classifier head."""
    model = models.efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)

    # Keep all layers trainable for the full fine-tuning.
    for parameter in model.parameters():
        parameter.requires_grad = True

    return model


def compute_class_weights(dataframe: pd.DataFrame) -> torch.Tensor:
    """Compute inverse-frequency class weights from a training dataframe."""
    counts = (
        dataframe["label"]
        .value_counts()
        .reindex(range(NUM_CLASSES), fill_value=0)
        .sort_index()
        .astype(float)
        .values
    )

    if np.any(counts == 0):
        missing_labels = [LABEL_TO_CLASS[i] for i, count in enumerate(counts) if count == 0]
        raise ValueError(f"Missing classes in the training fold: {missing_labels}")

    total_samples = counts.sum()
    class_weights = total_samples / (NUM_CLASSES * counts)
    return torch.tensor(class_weights, dtype=torch.float32)


MAX_EPOCHS = 30
PATIENCE = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

# Quick model check.
model_check = build_efficientnet_b0_model(NUM_CLASSES).to(DEVICE)
total_params = sum(p.numel() for p in model_check.parameters())
trainable_params = sum(p.numel() for p in model_check.parameters() if p.requires_grad)

# Show the class weights computed on the full dataset only as a diagnostic table.
# During cross-validation, the weights are recomputed inside each fold.
full_dataset_weights = compute_class_weights(dataset_df)
weights_table = pd.DataFrame(
    {
        "label": list(range(NUM_CLASSES)),
        "class_name": [LABEL_TO_CLASS[i] for i in range(NUM_CLASSES)],
        "full_dataset_weight_diagnostic": full_dataset_weights.numpy(),
    }
)

print(model_check.classifier)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print("Loss: weighted CrossEntropyLoss")
print(f"Optimizer: AdamW | lr={LEARNING_RATE} | weight_decay={WEIGHT_DECAY}")
print(f"Max epochs: {MAX_EPOCHS} | patience: {PATIENCE}")
display(weights_table)

del model_check
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


## 7. Training and validation functions

The loop follows the usual PyTorch sequence: forward pass, loss computation, backward pass and optimizer step. Balanced Accuracy is computed after each epoch because it is the main metric for this project.


In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, scaler, device, progress_bar=None):
    """Train the model for one epoch."""
    model.train()

    running_loss = 0.0
    all_targets = []
    all_predictions = []

    for images, targets in dataloader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        predictions = logits.argmax(dim=1)
        batch_size = images.size(0)

        running_loss += loss.item() * batch_size
        all_targets.extend(targets.detach().cpu().numpy())
        all_predictions.extend(predictions.detach().cpu().numpy())

        if progress_bar is not None:
            progress_bar.update(1)
            progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_accuracy = accuracy_score(all_targets, all_predictions)
    epoch_balanced_accuracy = balanced_accuracy_score(all_targets, all_predictions)

    return epoch_loss, epoch_accuracy, epoch_balanced_accuracy


@torch.inference_mode()
def evaluate(model, dataloader, criterion, device, progress_bar=None):
    """Evaluate the model without updating weights."""
    model.eval()

    running_loss = 0.0
    all_targets = []
    all_predictions = []

    for images, targets in dataloader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, targets)

        predictions = logits.argmax(dim=1)
        batch_size = images.size(0)

        running_loss += loss.item() * batch_size
        all_targets.extend(targets.detach().cpu().numpy())
        all_predictions.extend(predictions.detach().cpu().numpy())

        if progress_bar is not None:
            progress_bar.update(1)
            progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_accuracy = accuracy_score(all_targets, all_predictions)
    epoch_balanced_accuracy = balanced_accuracy_score(all_targets, all_predictions)

    return {
        "loss": epoch_loss,
        "accuracy": epoch_accuracy,
        "balanced_accuracy": epoch_balanced_accuracy,
        "targets": np.array(all_targets),
        "predictions": np.array(all_predictions),
    }


def copy_model_state_to_cpu(model: nn.Module) -> dict:
    """Copy the current model parameters to CPU without keeping GPU tensors."""
    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }


## 8. Five-fold cross-validation

Each fold starts from a new EfficientNet-B0 initialized with the same ImageNet weights. The best epoch of each fold is selected by validation Balanced Accuracy.


In [ ]:
def train_cv_fold(fold: int, train_indices, valid_indices):
    """Train one cross-validation fold and return its summary and best checkpoint."""

    # Fold-specific seed.
    fold_seed = SEED + fold

    random.seed(fold_seed)
    np.random.seed(fold_seed)
    torch.manual_seed(fold_seed)
    torch.cuda.manual_seed_all(fold_seed)

    fold_train_df = dataset_df.iloc[train_indices].reset_index(drop=True)
    fold_valid_df = dataset_df.iloc[valid_indices].reset_index(drop=True)

    fold_train_loader = make_loader(fold_train_df, train_transform, shuffle=True, seed=fold_seed)
    fold_valid_loader = make_loader(fold_valid_df, val_transform, shuffle=False)

    # Reinitialize model and training objects for this fold.
    model = build_efficientnet_b0_model(NUM_CLASSES).to(DEVICE)

    # Weighted CE uses weights computed only from the fold training subset.
    fold_class_weights = compute_class_weights(fold_train_df).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=fold_class_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2,
    )

    scaler = torch.amp.GradScaler(enabled=USE_AMP)

    history = []
    best_val_balanced_accuracy = -np.inf
    best_epoch = 0
    best_val_result = None
    best_checkpoint = None
    epochs_without_improvement = 0

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start_time = time.time()
        total_steps = len(fold_train_loader) + len(fold_valid_loader)

        with tqdm(
            total=total_steps,
            desc=f"Fold {fold}/{K_FOLDS} | epoch {epoch}/{MAX_EPOCHS}",
            dynamic_ncols=True,
            leave=True,
        ) as progress_bar:
            progress_bar.set_description(f"Fold {fold}/{K_FOLDS} | epoch {epoch} | train")
            train_loss, train_accuracy, train_balanced_accuracy = train_one_epoch(
                model=model,
                dataloader=fold_train_loader,
                criterion=criterion,
                optimizer=optimizer,
                scaler=scaler,
                device=DEVICE,
                progress_bar=progress_bar,
            )

            progress_bar.set_description(f"Fold {fold}/{K_FOLDS} | epoch {epoch} | val")
            val_result = evaluate(
                model=model,
                dataloader=fold_valid_loader,
                criterion=criterion,
                device=DEVICE,
                progress_bar=progress_bar,
            )

        scheduler.step(val_result["loss"])

        current_lr = optimizer.param_groups[0]["lr"]
        epoch_time = time.time() - epoch_start_time
        val_balanced_accuracy = val_result["balanced_accuracy"]
        improved = val_balanced_accuracy > best_val_balanced_accuracy

        if improved:
            best_val_balanced_accuracy = val_balanced_accuracy
            best_epoch = epoch
            best_val_result = val_result
            epochs_without_improvement = 0

            _, _, fold_macro_f1, _ = precision_recall_fscore_support(
                val_result["targets"],
                val_result["predictions"],
                average="macro",
                zero_division=0,
            )

            best_checkpoint = {
                "experiment_id": EXPERIMENT_ID,
                "fold": int(fold),
                "epoch": int(epoch),
                "model_state_dict": copy_model_state_to_cpu(model),
                "label_to_class": LABEL_TO_CLASS,
                "input_size": INPUT_SIZE,
                "imagenet_mean": IMAGENET_MEAN,
                "imagenet_std": IMAGENET_STD,
                "architecture": "efficientnet_b0",
                "loss": "WeightedCrossEntropyLoss",
                "class_weights": fold_class_weights.detach().cpu().tolist(),
                "training_strategy": "full_fine_tuning",
                "pretrained_weights": "IMAGENET1K_V1",
                "fold_val_loss": float(val_result["loss"]),
                "fold_val_accuracy": float(val_result["accuracy"]),
                "fold_val_balanced_accuracy": float(best_val_balanced_accuracy),
                "fold_val_macro_f1": float(fold_macro_f1),
                "selection_metric": "fold_val_balanced_accuracy",
            }
        else:
            epochs_without_improvement += 1

        gpu_memory_mb = None
        if DEVICE.type == "cuda":
            gpu_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

        history.append(
            {
                "fold": int(fold),
                "epoch": int(epoch),
                "lr": float(current_lr),
                "train_loss": float(train_loss),
                "train_accuracy": float(train_accuracy),
                "train_balanced_accuracy": float(train_balanced_accuracy),
                "fold_val_loss": float(val_result["loss"]),
                "fold_val_accuracy": float(val_result["accuracy"]),
                "fold_val_balanced_accuracy": float(val_balanced_accuracy),
                "epoch_time_sec": float(epoch_time),
                "gpu_memory_mb": gpu_memory_mb,
                "is_best": bool(improved),
            }
        )

        tqdm.write(
            f"Fold {fold}/{K_FOLDS} | "
            f"Epoch {epoch}/{MAX_EPOCHS} | "
            f"train_loss={train_loss:.4f} | "
            f"fold_val_loss={val_result['loss']:.4f} | "
            f"fold_val_bal_acc={val_balanced_accuracy:.4f} | "
            f"best={best_val_balanced_accuracy:.4f} @ {best_epoch} | "
            f"lr={current_lr:.1e} | "
            f"time={epoch_time:.1f}s"
        )

        if epochs_without_improvement >= PATIENCE:
            tqdm.write(f"Early stopping triggered for fold {fold} after {epoch} epochs.")
            break

    # Compute final fold metrics from the best epoch.
    precision, recall, f1, support = precision_recall_fscore_support(
        best_val_result["targets"],
        best_val_result["predictions"],
        labels=list(range(NUM_CLASSES)),
        zero_division=0,
    )

    summary_row = {
        "fold": int(fold),
        "train_images": int(len(fold_train_df)),
        "fold_val_images": int(len(fold_valid_df)),
        "best_epoch": int(best_epoch),
        "best_fold_val_loss": float(best_val_result["loss"]),
        "best_fold_val_accuracy": float(best_val_result["accuracy"]),
        "best_fold_val_balanced_accuracy": float(best_val_balanced_accuracy),
        "best_fold_val_macro_precision": float(np.mean(precision)),
        "best_fold_val_macro_recall": float(np.mean(recall)),
        "best_fold_val_macro_f1": float(np.mean(f1)),
        "class_weights": fold_class_weights.detach().cpu().tolist(),
    }

    history_df = pd.DataFrame(history)

    # Free GPU memory before the next fold.
    del model, optimizer, scheduler, criterion, scaler
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return summary_row, history_df, best_checkpoint


# Create stratified folds from the full dataset.
labels = dataset_df["label"].astype(int).values
stratified_kfold = StratifiedKFold(
    n_splits=K_FOLDS,
    shuffle=True,
    random_state=SEED,
)

cv_summary_rows = []
cv_history_frames = []
best_checkpoints = {}

cv_assignments_df = dataset_df.copy()
cv_assignments_df["cv_fold"] = -1

start_time = time.time()

for fold, (fold_train_indices, fold_valid_indices) in enumerate(
    stratified_kfold.split(dataset_df, labels),
    start=1,
):
    print(f"\n========== Fold {fold}/{K_FOLDS} ==========")
    cv_assignments_df.loc[fold_valid_indices, "cv_fold"] = fold

    summary_row, fold_history_df, best_checkpoint = train_cv_fold(
        fold,
        fold_train_indices,
        fold_valid_indices,
    )

    cv_summary_rows.append(summary_row)
    cv_history_frames.append(fold_history_df)
    best_checkpoints[fold] = best_checkpoint

cv_summary_df = pd.DataFrame(cv_summary_rows)
cv_history_df = pd.concat(cv_history_frames, ignore_index=True)

elapsed_minutes = (time.time() - start_time) / 60

print("\nCross-validation summary:")
display(cv_summary_df)

print(
    "Mean CV balanced accuracy: "
    f"{cv_summary_df['best_fold_val_balanced_accuracy'].mean():.4f} "
    f"+/- {cv_summary_df['best_fold_val_balanced_accuracy'].std(ddof=1):.4f}"
)
print(f"Cross-validation time: {elapsed_minutes:.1f} minutes")


## 9. Representative fold

We select the fold whose best Balanced Accuracy is closest to the cross-validation mean. This model is used only to inspect the normalized confusion matrix on its validation fold.


In [ ]:
cv_mean_balanced_accuracy = float(cv_summary_df["best_fold_val_balanced_accuracy"].mean())
cv_std_balanced_accuracy = float(cv_summary_df["best_fold_val_balanced_accuracy"].std(ddof=1))

# Select the fold closest to the CV mean.
cv_summary_df["distance_from_mean_balanced_accuracy"] = (
    cv_summary_df["best_fold_val_balanced_accuracy"] - cv_mean_balanced_accuracy
).abs()

selected_row = cv_summary_df.sort_values(
    by=["distance_from_mean_balanced_accuracy", "best_fold_val_balanced_accuracy"],
    ascending=[True, False],
).iloc[0]

selected_fold = int(selected_row["fold"])
selected_checkpoint = best_checkpoints[selected_fold]

selected_metadata = {
    "selected_fold": selected_fold,
    "selection_rule": "closest_to_cv_mean_balanced_accuracy",
    "cv_mean_balanced_accuracy": cv_mean_balanced_accuracy,
    "cv_std_balanced_accuracy": cv_std_balanced_accuracy,
    "selected_fold_val_balanced_accuracy": float(selected_row["best_fold_val_balanced_accuracy"]),
    "distance_from_mean_balanced_accuracy": float(selected_row["distance_from_mean_balanced_accuracy"]),
    "selected_best_epoch": int(selected_row["best_epoch"]),
}

print(json.dumps(selected_metadata, indent=4))


## 10. Normalized confusion matrix

The selected fold is evaluated again to build the normalized confusion matrix. Each row shows how the images of one true class are distributed across the predicted labels.


In [ ]:
# Recreate the validation fold used by the selected checkpoint.
selected_valid_df = cv_assignments_df[cv_assignments_df["cv_fold"] == selected_fold].reset_index(drop=True)
selected_valid_loader = make_loader(selected_valid_df, val_transform, shuffle=False)

# Load selected checkpoint in memory.
model = build_efficientnet_b0_model(NUM_CLASSES).to(DEVICE)
model.load_state_dict(selected_checkpoint["model_state_dict"])

selected_class_weights = torch.tensor(
    selected_checkpoint["class_weights"],
    dtype=torch.float32,
    device=DEVICE,
)
criterion = nn.CrossEntropyLoss(weight=selected_class_weights)

selected_result = evaluate(
    model=model,
    dataloader=selected_valid_loader,
    criterion=criterion,
    device=DEVICE,
)

y_true = selected_result["targets"]
y_pred = selected_result["predictions"]

# Normalized confusion matrix.
cm_raw = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
cm_norm = np.divide(
    cm_raw,
    cm_raw.sum(axis=1, keepdims=True),
    out=np.zeros_like(cm_raw, dtype=float),
    where=cm_raw.sum(axis=1, keepdims=True) != 0,
)

cm_norm_df = pd.DataFrame(
    cm_norm,
    index=[LABEL_TO_CLASS[i] for i in range(NUM_CLASSES)],
    columns=[LABEL_TO_CLASS[i] for i in range(NUM_CLASSES)],
)

cm_norm_df.to_csv(CONFUSION_MATRIX_NORM_PATH)

# Plot normalized confusion matrix.
plt.figure(figsize=(8, 7))
plt.imshow(cm_norm, interpolation="nearest", vmin=0.0, vmax=1.0)
plt.title("Selected fold normalized confusion matrix")
plt.xticks(range(NUM_CLASSES), [LABEL_TO_CLASS[i] for i in range(NUM_CLASSES)], rotation=45, ha="right")
plt.yticks(range(NUM_CLASSES), [LABEL_TO_CLASS[i] for i in range(NUM_CLASSES)])
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.colorbar()

for row in range(NUM_CLASSES):
    for col in range(NUM_CLASSES):
        plt.text(col, row, f"{cm_norm[row, col]:.2f}", ha="center", va="center")

plt.tight_layout()
plt.savefig(CONFUSION_MATRIX_FIG_PATH, dpi=150)
plt.show()

display(cm_norm_df)
print(f"Saved normalized confusion matrix CSV: {CONFUSION_MATRIX_NORM_PATH}")
print(f"Saved normalized confusion matrix figure: {CONFUSION_MATRIX_FIG_PATH}")


## 11. Final summary

The summary stores the main training setup, cross-validation scores and selected-fold metrics.


In [ ]:
selected_precision, selected_recall, selected_f1, selected_support = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES)),
    zero_division=0,
)

selected_metrics = {
    "selected_val_loss": float(selected_result["loss"]),
    "selected_val_accuracy": float(selected_result["accuracy"]),
    "selected_val_balanced_accuracy": float(selected_result["balanced_accuracy"]),
    "macro_precision": float(np.mean(selected_precision)),
    "macro_recall": float(np.mean(selected_recall)),
    "macro_f1": float(np.mean(selected_f1)),
}

summary = {
    "experiment_id": EXPERIMENT_ID,
    "dataset_images": int(len(dataset_df)),
    "class_distribution": dataset_distribution_df.to_dict(orient="records"),
    "architecture": "efficientnet_b0",
    "pretrained_weights": "IMAGENET1K_V1",
    "training_strategy": "full_fine_tuning",
    "loss": "WeightedCrossEntropyLoss",
    "selection_metric": "fold_val_balanced_accuracy",
    "k_folds": int(K_FOLDS),
    "seed": int(SEED),
    "input_size": int(INPUT_SIZE),
    "batch_size": int(BATCH_SIZE),
    "num_workers": int(NUM_WORKERS),
    "max_epochs": int(MAX_EPOCHS),
    "patience": int(PATIENCE),
    "learning_rate": float(LEARNING_RATE),
    "weight_decay": float(WEIGHT_DECAY),
    "augmentation": "disabled",
    "cv_mean_balanced_accuracy": float(cv_mean_balanced_accuracy),
    "cv_std_balanced_accuracy": float(cv_std_balanced_accuracy),
    "cv_summary": cv_summary_df.drop(columns=["class_weights"]).to_dict(orient="records"),
    "selected_fold": int(selected_fold),
    "selected_best_epoch": int(selected_metadata["selected_best_epoch"]),
    "selected_metrics": selected_metrics,
    "outputs": {
        "summary": str(SUMMARY_PATH),
        "normalized_confusion_matrix_csv": str(CONFUSION_MATRIX_NORM_PATH),
        "normalized_confusion_matrix_figure": str(CONFUSION_MATRIX_FIG_PATH),
    },
}

with SUMMARY_PATH.open("w") as file:
    json.dump(summary, file, indent=4)

compact_summary_df = pd.DataFrame(
    [
        {
            "experiment_id": EXPERIMENT_ID,
            "dataset_images": int(len(dataset_df)),
            "cv_mean_balanced_accuracy": float(cv_mean_balanced_accuracy),
            "cv_std_balanced_accuracy": float(cv_std_balanced_accuracy),
            "selected_fold": int(selected_fold),
            "selected_best_epoch": int(selected_metadata["selected_best_epoch"]),
            **selected_metrics,
        }
    ]
)

display(compact_summary_df)
print(f"Saved summary: {SUMMARY_PATH}")
